# Study 868 — Global Curve-Slope Carry 🌍

**Does a *steep* yield curve pay a duration holder — long the high-carry, short the flat markets?**

Koijen, Moskowitz, Pedersen & Vrugt (2018) *"Carry"* find carry predicts returns across
asset classes, including bonds: a steep curve pays a holder the yield *and* the roll-down.
We take the sovereign-bond sleeve on tradable ground — six US + international government-bond
ETFs (`SHY`, `IEF`, `TLT`, `BWX`, `IGOV`, `BNDX`), ranked each month by a **yield-to-duration
carry proxy**, long the high-carry / short the low-carry markets, dollar-neutral,
2007-01-31 → 2026-06-30.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: currently-listed funds only, a short full cross-section
(BNDX from 2013), and a price-only carry proxy — an upper bound with limited power.*


## 1. The idea in one picture

A **steep** curve is supposed to pay a duration holder twice — a higher yield *and* a capital gain as the bond ages and rolls *down* toward lower yields. So across bond markets a duration investor should tilt toward the steep-curve / high-carry sleeves and away from the flat / low-carry ones. We proxy each ETF's carry by its trailing realized yield ÷ its duration and sort the markets on it.

In [1]:
R = dict(ytd_bps=-20.17, ytd_t_nw=-1.45, raw_bps=3.16, raw_t_nw=0.2, bh_bps=21.36, bh_t=1.79)
print('YIELD-TO-DURATION carry sort: %+.2f bps/mo (Newey-West t = %+.2f)' % (R['ytd_bps'], R['ytd_t_nw']))
print('RAW realized-yield carry sort: %+.2f bps/mo (Newey-West t = %+.2f)' % (R['raw_bps'], R['raw_t_nw']))
print('NAIVE buy-and-hold           : %+.2f bps/mo (Newey-West t = %+.2f)  <- the real yardstick'
      % (R['bh_bps'], R['bh_t']))

YIELD-TO-DURATION carry sort: -20.17 bps/mo (Newey-West t = -1.45)
RAW realized-yield carry sort: +3.16 bps/mo (Newey-West t = +0.20)
NAIVE buy-and-hold           : +21.36 bps/mo (Newey-West t = +1.79)  <- the real yardstick


## 2. Is the carry real? A live synthetic control

We plant a fixed structural carry spread in a seeded toy world (`edge>0`) and check the detector recovers it — and that it stays *silent* when every market yields the same (`edge=0`, nothing to sort on). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from curve_slope_carry import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=868))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.010, seed=868))
print('equal-yield world  : NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted-carry world: NW t = %+.2f  (should light up)' % planted['t_nw'])

equal-yield world  : NW t = -0.42  (should be ~0)
planted-carry world: NW t = +18.09  (should light up)


## 3. The honest verdict — the carry sort does *not* pay here

On six sovereign-bond ETFs the **yield-to-duration** carry sort earns the **wrong sign**: **-20.17 bps/mo**, Newey-West *t* = **-1.45** — its 'low-carry' short leg (+30.16 bps) actually *out-earns* its 'high-carry' long leg (+9.99 bps), and a random leg assignment beats it 94% of the time. The plainer **raw realized-yield** sort is **dead flat** (**+3.16 bps/mo**, NW *t* = **+0.20**). Naive equal-weight buy-and-hold earns **+21.36 bps/mo** — *more* than either timed book. **Signal: None** (claimed edge absent, if anything perversely negative), **Tradability: Mirage** (every variant loses money after costs).